<a href="https://colab.research.google.com/github/TBGhorbanpour/Social-Awareness/blob/main/Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd
import torch
from transformers import pipeline
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# ==========================================
# 1. Configuration & Device Check
# ==========================================
# Use the latest cleaned CSV from your previous step
INPUT_CSV = '/content/drive/MyDrive/Thesis/Data/Data_Paper/MainDB_NER_Cleaned.csv'
OUTPUT_CSV = '/content/drive/MyDrive/Thesis/Data/Data_Paper/MainDB_Sentiment_Output.csv'

# Force GPU usage if available (Colab T4/A100)
device = 0 if torch.cuda.is_available() else -1
print(f"🚀 Using device: {'GPU (CUDA)' if device == 0 else 'CPU (Warning: This will be very slow!)'}")

# ==========================================
# 2. Load Data
# ==========================================
print("Loading dataset...")
data = pd.read_csv(INPUT_CSV)

# Ensure we have clean strings and handle NaNs safely
# Using 'bert_ready_tweets' as requested
texts = data['bert_ready_tweets'].fillna('').astype(str).tolist()
print(f"Loaded {len(texts)} tweets for sentiment analysis.")

# ==========================================
# 3. Initialize Batched Sentiment Pipeline
# ==========================================
print("Loading ParsBERT Sentiment model...")
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="HooshvareLab/bert-fa-base-uncased-sentiment-deepsentipers-multi",
    tokenizer="HooshvareLab/bert-fa-base-uncased-sentiment-deepsentipers-multi",
    device=device,
    batch_size=16  # Process 16 tweets at a time (Lower to 8 if you get CUDA Out of Memory)
    # Note: truncation/max_length omitted to prevent the _sanitize_parameters TypeError.
    # Tweets are naturally short and will safely fit within BERT's default 512 limit.
)

# ==========================================
# 4. Run Inference (Fast!)
# ==========================================
print("Starting batched sentiment analysis... (This will take ~10-15 mins on GPU)")
sentiment_results = sentiment_pipeline(texts)

# ==========================================
# 5. Format and Save Results
# ==========================================
print("Formatting results...")

# The pipeline returns a list of dicts: [{'label': 'positive', 'score': 0.95}, ...]
# We extract these directly into two new columns efficiently using list comprehensions
data['sentiment_label'] = [res['label'] for res in sentiment_results]
data['confidence_score'] = [res['score'] for res in sentiment_results]

# Save to Drive
data.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f"✅ Done! Saved sentiment results to {OUTPUT_CSV}")

# Display Preview and Distribution
print("\n📊 Sentiment Distribution:")
print(data['sentiment_label'].value_counts())

print("\n👀 Preview:")
display_cols = ['bert_ready_tweets', 'sentiment_label', 'confidence_score']
display(data[display_cols].head(5))

🚀 Using device: GPU (CUDA)
Loading dataset...
Loaded 191245 tweets for sentiment analysis.
Loading ParsBERT Sentiment model...


config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  651MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

model.safetensors: downloading bytes:           |  0.00B            

vocab.txt:   0%|          | 0.00/1.20M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Starting batched sentiment analysis... (This will take ~10-15 mins on GPU)
Formatting results...
✅ Done! Saved sentiment results to /content/drive/MyDrive/Thesis/Data/download data folder ff/MainDB_Sentiment_Output.csv

📊 Sentiment Distribution:
sentiment_label
neutral      79043
furious      59294
happy        20607
angry        19419
delighted    12882
Name: count, dtype: int64

👀 Preview:


,bert_ready_tweets,sentiment_label,confidence_score
0,همان که خاک هرمز بود دیگر؟!,neutral,0.994297
1,تاحالا نان توموشی به کسی که اولین بارش آمده بن...,delighted,0.833349
2,سرنج به معنی خاک سرخ وایب هرمز گرفتم از تو)),neutral,0.996368
3,یکی از شگفت‌انگیزترین ساحل‌های ایران که شهرت ج...,delighted,0.558333
4,خاک‌های رنگی جزیره هرمز,neutral,0.909420


In [9]:
import pandas as pd
import ast

data = pd.read_csv('/content/drive/MyDrive/Thesis/Data/Data_Paper/MainDB_Sentiment_Output.csv')

# Convert string back to list
data['cities_list'] = data['cities'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# Explode to see individual extracted cities
all_extracted_cities = data['cities_list'].explode().value_counts()

# Filter for anything containing "زاینده"
zayandeh_variations = all_extracted_cities[all_extracted_cities.index.str.contains('زاینده', na=False)]
print("Current Zayandeh Rood variations extracted by NER:")
print(zayandeh_variations)

Current Zayandeh Rood variations extracted by NER:
cities_list
زایندهرود                                  3914
اصفهان زایندهرود                           1298
زاینده _                                    786
زایندهرود اصفهان                            752
اصفهان زاینده _                             719
                                           ... 
باغ موزه اصفهان زایندهرود اصفهان              1
بروجن زایندهرود اصفهان                        1
زایندهرود خشکه رود زاینده صحرا اصفهان         1
اصفهان زایندهرود ایران اصفهان زایندهرود       1
کره شمالی اصفهان زاینده _ ایران               1
Name: count, Length: 6448, dtype: int64


In [15]:
import pandas as pd
import ast

# ==========================================
# 1. Configuration & Load Data
# ==========================================
INPUT_CSV = '/content/drive/MyDrive/Thesis/Data/Data_Paper/MainDB_Sentiment_Output.csv'
OUTPUT_SUMMARY_CSV = '/content/drive/MyDrive/Thesis/Data/Data_Paper/City_Sentiment_Summary.csv'
OUTPUT_MAPPED_DATA_CSV = '/content/drive/MyDrive/Thesis/Data/Data_Paper/MainDB_Final_Mapped.csv'

print("Loading dataset...")
data = pd.read_csv(INPUT_CSV)

# ==========================================
# 2. Location Mapping Dictionary
# ==========================================
location_mapping = {
    # Urmia
    'ارومیه': 'ارومیه', 'اورمیه': 'ارومیه', 'ارومیه دریاچه': 'ارومیه',
    'دریاچه ارومیه': 'ارومیه', 'دریاچه_ارومیه': 'ارومیه', 'دریاچه _ ارومیه': 'ارومیه',
    'اورمیه دریاچه': 'ارومیه', 'تالاب ارومیه': 'ارومیه', 'تالاب_ارومیه': 'ارومیه',
    'دریاچه ارومی': 'ارومیه', 'رضاییه': 'ارومیه',

    # Isfahan
    'اصفهان': 'اصفهان', 'ا_صفهان': 'اصفهان', 'اصف_هان': 'اصفهان', 'استان اصفهان': 'اصفهان',
    'سپاهان': 'اصفهان', 'اصفهان نصف جهان': 'اصفهان', 'نصف جهان': 'اصفهان',
    'زایندهرود': 'اصفهان', 'زاینده_رود': 'اصفهان', 'زاینده _': 'اصفهان',
    'زاینده رود': 'اصفهان', 'زاینده': 'اصفهان', 'رود زاینده': 'اصفهان',
    'اصفهان زایندهرود': 'اصفهان', 'زایندهرود اصفهان': 'اصفهان', 'اصفهان زاینده _': 'اصفهان',
    'زایندهرود خشک': 'اصفهان', 'زاینده رود خشک': 'اصفهان',
    'میدان نقش جهان': 'اصفهان', 'نقش جهان': 'اصفهان', 'نقش_جهان': 'اصفهان',
    'سی و سه پل': 'اصفهان', 'سی و سه_پل': 'اصفهان', 'سی و سه پل اصفهان': 'اصفهان',
    'پل خواجو': 'اصفهان', 'خواجو': 'اصفهان', 'چهارباغ': 'اصفهان', 'چهارباغ عباسی': 'اصفهان',
    'کوه صفه': 'اصفهان', 'صفه': 'اصفهان',
    'کاشان': 'اصفهان', 'نائین': 'اصفهان', 'خمینی شهر': 'اصفهان', 'خمینی‌شهر': 'اصفهان',
    'نجف آباد': 'اصفهان', 'نجف‌آباد': 'اصفهان', 'شاهین شهر': 'اصفهان', 'شاهین‌شهر': 'اصفهان',
    'مبارکه': 'اصفهان', 'فولادشهر': 'اصفهان', 'لنجان': 'اصفهان', 'شهرضا': 'اصفهان',

    # Khuzestan
    'خوزستان': 'خوزستان', 'خوز_ستان': 'خوزستان', 'استان خوزستان': 'خوزستان',
    'اهواز': 'خوزستان', 'کارون': 'خوزستان', 'کرخه': 'خوزستان',
    'آبادان': 'خوزستان', 'خرمشهر': 'خوزستان', 'دزفول': 'خوزستان',
    'ماهشهر': 'خوزستان', 'بندر ماهشهر': 'خوزستان', 'پتروشیمی ماهشهر': 'خوزستان',
    'شادگان': 'خوزستان', 'تالاب شادگان': 'خوزستان',
    'هویزه': 'خوزستان', 'تالاب هویزه': 'خوزستان',
    'شوش': 'خوزستان', 'شوشتر': 'خوزستان', 'مسجدسلیمان': 'خوزستان',
    'بهبهان': 'خوزستان', 'ایذه': 'خوزستان', 'رامهرمز': 'خوزستان',
    'لالی': 'خوزستان', 'هفتکل': 'خوزستان', 'گتوند': 'خوزستان', 'سردشت': 'خوزستان',

    # Tehran
    'تهران': 'تهران', 'طهران': 'تهران', 'ت_هران': 'تهران', 'استان تهران': 'تهران',
    'کرج': 'تهران', 'شهرری': 'تهران', 'ری': 'تهران', 'اسلامشهر': 'تهران',
    'ورامین': 'تهران', 'دماوند': 'تهران', 'فیروزکوه': 'تهران',
    'تجریش': 'تهران', 'ونک': 'تهران', 'سعادت آباد': 'تهران', 'پونک': 'تهران',
    'زعفرانیه': 'تهران', 'الهیه': 'تهران', 'اقدسیه': 'تهران', 'درکه': 'تهران', 'دربند': 'تهران',
    'توچال': 'تهران', 'برج میلاد': 'تهران', 'میلاد': 'تهران',
    'برج آزادی': 'تهران', 'میدان آزادی': 'تهران',
    'میدان انقلاب': 'تهران', 'خیابان انقلاب': 'تهران', 'میدان ولیعصر': 'تهران', 'ولیعصر': 'تهران',
    'دانشگاه تهران': 'تهران', 'دانشگاه شریف': 'تهران', 'دانشگاه بهشتی': 'تهران',
    'مهرآباد': 'تهران', 'فرودگاه مهرآباد': 'تهران', 'امام خمینی': 'تهران', 'فرودگاه امام': 'تهران'
}

# ==========================================
# 3. Apply Mapping
# ==========================================
def safe_eval(val):
    if isinstance(val, str):
        try:
            return ast.literal_eval(val)
        except (ValueError, SyntaxError):
            return []
    return val if isinstance(val, list) else []

def map_cities(city_list):
    if not isinstance(city_list, list):
        return []
    mapped = []
    for city in city_list:
        city = str(city).strip()
        mapped.append(location_mapping.get(city, city)) # Map if exists, else keep original
    return list(set(mapped)) # Remove duplicates within the same tweet

print("Applying location mapping...")
data['cities_list'] = data['cities'].apply(safe_eval)
data['mapped_cities'] = data['cities_list'].apply(map_cities)

# ==========================================
# 4. Calculate Sentiment Statistics
# ==========================================
print("Calculating sentiment statistics...")
exploded = data.explode('mapped_cities')
exploded = exploded[exploded['mapped_cities'].notna() & (exploded['mapped_cities'] != '')]

# Group by City and Sentiment to get counts
stats = exploded.groupby(['mapped_cities', 'sentiment_label']).size().reset_index(name='Count')

# Calculate total tweets per city to get percentages
city_totals = stats.groupby('mapped_cities')['Count'].sum().reset_index(name='Total_Tweets')
stats = stats.merge(city_totals, on='mapped_cities')
stats['Percentage'] = (stats['Count'] / stats['Total_Tweets'] * 100).round(1)

# Rename column for clarity
stats.rename(columns={'mapped_cities': 'City', 'sentiment_label': 'Sentiment'}, inplace=True)

# Sort by Total Tweets (descending) then by Count (descending)
stats = stats.sort_values(by=['Total_Tweets', 'Count'], ascending=[False, False])

# ==========================================
# 5. Save to CSV
# ==========================================
# Save the summary statistics
stats.to_csv(OUTPUT_SUMMARY_CSV, index=False, encoding='utf-8-sig')
print(f"✅ Sentiment summary saved to: {OUTPUT_SUMMARY_CSV}")

# Optional: Save the fully mapped row-level data for future analysis
data.to_csv(OUTPUT_MAPPED_DATA_CSV, index=False, encoding='utf-8-sig')
print(f"✅ Fully mapped dataset saved to: {OUTPUT_MAPPED_DATA_CSV}")

# ==========================================
# 6. Display Preview
# ==========================================
print("\n📊 Top 20 Rows of Sentiment Summary:")
print(stats.head(20).to_string(index=False))

Loading dataset...
Applying location mapping...
Calculating sentiment statistics...
✅ Sentiment summary saved to: /content/drive/MyDrive/Thesis/Data/download data folder ff/City_Sentiment_Summary.csv
✅ Fully mapped dataset saved to: /content/drive/MyDrive/Thesis/Data/download data folder ff/MainDB_Final_Mapped.csv

📊 Top 20 Rows of Sentiment Summary:
   City Sentiment  Count  Total_Tweets  Percentage
 اصفهان   neutral   5881         11066        53.1
 اصفهان   furious   2493         11066        22.5
 اصفهان     angry   1170         11066        10.6
 اصفهان     happy    895         11066         8.1
 اصفهان delighted    627         11066         5.7
  ایران   furious   4198         10571        39.7
  ایران   neutral   3953         10571        37.4
  ایران     happy    940         10571         8.9
  ایران delighted    818         10571         7.7
  ایران     angry    662         10571         6.3
 ارومیه   neutral   4325          8150        53.1
 ارومیه   furious   1704          8